# Memento — Layered World Visualization
Modular notebook with clear chapter separation: **load**, **validate**, **compute**, **build canvas**, and **add layers**. Worlds are discovered dynamically and rendered as one figure per world.

## Chapter 1 — Load Raw Data
Load snapshot CSV and validate that required columns exist.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------
# DATA_FILE = '../data/memento_world_snapshot.large.csv'
DATA_FILE = '../data/memento_world_snapshot.small.csv'

REQUIRED_COLUMNS = [
    'dimension', 'regionX', 'regionZ', 'chunkX', 'chunkZ', 'scanTick',
    'inhabitedTicks', 'surfaceY', 'biome', 'isSpawn',
    'dominantStone', 'dominantStoneEffect',
    'chunkMemorable', 'chunkForgettable',
    'renewalAction', 'renewalRank', 'source', 'status'
]

def load_raw_data(path: str) -> pd.DataFrame:
    """Load raw CSV snapshot."""
    return pd.read_csv(path)

def validate_required_columns(df: pd.DataFrame, required_columns: list[str]) -> None:
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

df_raw = load_raw_data(DATA_FILE)
validate_required_columns(df_raw, REQUIRED_COLUMNS)

print(f'Rows: {len(df_raw):,}')
print(f'Worlds discovered: {df_raw["dimension"].nunique()}')
print('World keys:', sorted(df_raw['dimension'].unique().tolist()))

## Chapter 2 — Sanity Checks
Run data integrity checks before deriving computed fields.

In [ ]:
# ----------------------------------------------------------
# Sanity checks
# ----------------------------------------------------------

def check_unique_coords(df: pd.DataFrame) -> int:
    """Count duplicates by absolute chunk coordinates per dimension."""
    abs_x = pd.to_numeric(df['regionX'], errors='coerce').fillna(0).astype(int) * 32 + pd.to_numeric(df['chunkX'], errors='coerce').fillna(0).astype(int)
    abs_z = pd.to_numeric(df['regionZ'], errors='coerce').fillna(0).astype(int) * 32 + pd.to_numeric(df['chunkZ'], errors='coerce').fillna(0).astype(int)
    key = pd.DataFrame({'dimension': df['dimension'], 'abs_x': abs_x, 'abs_z': abs_z})
    return int(key.duplicated(subset=['dimension', 'abs_x', 'abs_z']).sum())

def check_mutual_exclusive_flags(df: pd.DataFrame) -> int:
    """Rows where chunkMemorable and chunkForgettable are both 1."""
    mem = pd.to_numeric(df['chunkMemorable'], errors='coerce').fillna(0).astype(int)
    forget = pd.to_numeric(df['chunkForgettable'], errors='coerce').fillna(0).astype(int)
    return int(((mem == 1) & (forget == 1)).sum())

def check_status_ok(df: pd.DataFrame) -> int:
    """Rows where status is not OK."""
    return int((df['status'].fillna('') != 'OK').sum())

def run_sanity_checks(df: pd.DataFrame) -> dict[str, int]:
    results = {
        'duplicate_absolute_coords': check_unique_coords(df),
        'memorable_and_forgettable': check_mutual_exclusive_flags(df),
        'status_not_ok': check_status_ok(df),
    }

    failed = {k: v for k, v in results.items() if v != 0}
    if failed:
        raise ValueError(f'Sanity checks failed: {failed}')

    return results

sanity = run_sanity_checks(df_raw)
print('Sanity checks passed:', sanity)

## Chapter 3 — Compute All Data (Per World)
Create a dynamic world map and prepare each world dataset with normalized and cleaned fields.

In [ ]:
# ----------------------------------------------------------
# Data algorithms (world-by-world)
# ----------------------------------------------------------

def normalize_series(series: pd.Series) -> pd.Series:
    min_v = series.min()
    max_v = series.max()
    if pd.isna(min_v) or pd.isna(max_v) or max_v == min_v:
        return pd.Series(np.zeros(len(series), dtype=float), index=series.index)
    return (series - min_v) / (max_v - min_v)

def prepare_world_dataset(world_df: pd.DataFrame) -> pd.DataFrame:
    d = world_df.copy()

    # standard axis coordinates (absolute chunk coordinates)
    region_x = pd.to_numeric(d['regionX'], errors='coerce').fillna(0).astype(int)
    region_z = pd.to_numeric(d['regionZ'], errors='coerce').fillna(0).astype(int)
    local_x = pd.to_numeric(d['chunkX'], errors='coerce').fillna(0).astype(int)
    local_z = pd.to_numeric(d['chunkZ'], errors='coerce').fillna(0).astype(int)
    d['x'] = region_x * 32 + local_x
    d['z'] = region_z * 32 + local_z

    # booleans / flags
    d['forgettableFlag'] = pd.to_numeric(d['chunkForgettable'], errors='coerce').fillna(0).astype(int)
    d['memorableFlag'] = pd.to_numeric(d['chunkMemorable'], errors='coerce').fillna(0).astype(int)

    # numeric fields
    d['inhabitedTicks'] = pd.to_numeric(d['inhabitedTicks'], errors='coerce').fillna(0)
    d['renewalRank'] = pd.to_numeric(d['renewalRank'], errors='coerce')

    # per-world calibration
    d['inhabitance_norm'] = normalize_series(d['inhabitedTicks'])

    # categoricals
    d['renewalAction'] = d['renewalAction'].fillna('NONE').astype(str)
    d['dominantStoneEffect'] = d['dominantStoneEffect'].fillna('NONE').astype(str)

    return d

def build_world_datasets(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    worlds: dict[str, pd.DataFrame] = {}
    for world_name, world_df in df.groupby('dimension', sort=True):
        worlds[world_name] = prepare_world_dataset(world_df)
    return worlds

WORLD_DATASETS = build_world_datasets(df_raw)

# Global category sets to keep legends consistent across worlds
GLOBAL_RENEWAL_ACTIONS = sorted([x for x in df_raw['renewalAction'].fillna('NONE').astype(str).unique().tolist() if x != 'NONE'])
GLOBAL_STONE_EFFECTS = sorted([x for x in df_raw['dominantStoneEffect'].fillna('NONE').astype(str).unique().tolist() if x != 'NONE'])

print(f'Total worlds: {len(WORLD_DATASETS)}')
for world_name, world_df in WORLD_DATASETS.items():
    print(f'- {world_name}: {len(world_df):,} chunks')
print('Global renewalAction categories:', GLOBAL_RENEWAL_ACTIONS if GLOBAL_RENEWAL_ACTIONS else ['<none>'])
print('Global dominantStoneEffect categories:', GLOBAL_STONE_EFFECTS if GLOBAL_STONE_EFFECTS else ['<none>'])

## Chapter 4 — Prepare Plotly Canvas
Create reusable figure canvas with improved legend readability and room allocation.

In [ ]:
# ----------------------------------------------------------
# Visualization infrastructure
# ----------------------------------------------------------

def build_world_grids(world_df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    return {
        'inhabitance': world_df.pivot_table(index='z', columns='x', values='inhabitance_norm', aggfunc='first').sort_index(),
        'forgettable': world_df.pivot_table(index='z', columns='x', values='forgettableFlag', aggfunc='first').sort_index(),
        'memorable': world_df.pivot_table(index='z', columns='x', values='memorableFlag', aggfunc='first').sort_index(),
        'renewalRank': world_df.pivot_table(index='z', columns='x', values='renewalRank', aggfunc='first').sort_index(),
    }

def build_canvas(world_name: str, world_df: pd.DataFrame) -> go.Figure:
    fig = go.Figure()

    x_min, x_max = int(world_df['x'].min()), int(world_df['x'].max())
    z_min, z_max = int(world_df['z'].min()), int(world_df['z'].max())

    fig.update_layout(
        title=f'Memento World Layers — {world_name}',
        template='plotly_white',
        hovermode='closest',
        xaxis=dict(
            title='Chunk X',
            showgrid=False,
            gridcolor='rgba(180, 180, 180, 0.20)',
            zeroline=False,
            range=[x_min - 0.5, x_max + 0.5],
        ),
        yaxis=dict(
            title='Chunk Z',
            showgrid=False,
            gridcolor='rgba(180, 180, 180, 0.20)',
            zeroline=False,
            scaleanchor='x',
            scaleratio=1,
            range=[z_min - 0.5, z_max + 0.5],
        ),
        # Legend on right, colorbars on left to avoid overlap
        legend=dict(
            orientation='v',
            yanchor='top',
            y=1.0,
            xanchor='left',
            x=1.02,
            bgcolor='rgba(255,255,255,0.92)',
            bordercolor='rgba(80,80,80,0.35)',
            borderwidth=1,
            font=dict(size=14),
            tracegroupgap=8,
            itemsizing='constant',
        ),
        margin=dict(l=170, r=300),
    )

    return fig

## Chapter 5 — Build Visualization Layer by Layer
Each layer is a dedicated helper function. Legend is the only interaction control for toggling layers.

In [ ]:
# ----------------------------------------------------------
# Layer functions
# ----------------------------------------------------------

def _binary_mask(grid: pd.DataFrame) -> np.ndarray:
    arr = grid.values.astype(float)
    return np.where(arr >= 1, 1.0, np.nan)

def _add_legend_placeholder(fig: go.Figure, name: str, legendgroup: str, color: str = 'rgba(120,120,120,0.9)') -> None:
    """Add a no-data placeholder so legend entries stay consistent across worlds."""
    fig.add_trace(go.Scattergl(
        x=[None], y=[None],
        mode='markers',
        name=name,
        legendgroup=legendgroup,
        visible='legendonly',
        showlegend=True,
        marker=dict(symbol='square', size=8, color=color),
        hoverinfo='skip'
    ))

def add_layer_chunk_exists(fig: go.Figure, world_df: pd.DataFrame, visible: bool | str = 'legendonly') -> None:
    fig.add_trace(go.Scattergl(
        x=world_df['x'],
        y=world_df['z'],
        mode='markers',
        name='chunkExists',
        legendgroup='chunkExists',
        visible=visible,
        showlegend=True,
        marker=dict(symbol='square', size=6, color='rgba(120,120,120,0.28)'),
        hovertemplate='Chunk (%{x}, %{y})<extra>chunkExists</extra>',
    ))


def add_layer_region_grid(fig: go.Figure, world_df: pd.DataFrame, visible: bool | str = True) -> None:
    x_min, x_max = int(world_df['x'].min()), int(world_df['x'].max())
    z_min, z_max = int(world_df['z'].min()), int(world_df['z'].max())

    region_x_start = (x_min // 32) * 32
    region_x_end = ((x_max + 31) // 32) * 32
    region_z_start = (z_min // 32) * 32
    region_z_end = ((z_max + 31) // 32) * 32

    fine_x, fine_z = [], []

    for x in range(region_x_start, region_x_end + 1, 32):
        if x == 0:
            continue
        fine_x.extend([x, x, None])
        fine_z.extend([z_min - 0.5, z_max + 0.5, None])

    for z in range(region_z_start, region_z_end + 1, 32):
        if z == 0:
            continue
        fine_x.extend([x_min - 0.5, x_max + 0.5, None])
        fine_z.extend([z, z, None])

    fig.add_trace(go.Scattergl(
        x=fine_x,
        y=fine_z,
        mode='lines',
        name='regionGrid',
        legendgroup='regionGrid',
        visible=visible,
        showlegend=True,
        line=dict(color='rgba(80,80,80,0.35)', width=1),
        hoverinfo='skip',
    ))

    fig.add_trace(go.Scattergl(
        x=[0, 0, None, x_min - 0.5, x_max + 0.5],
        y=[z_min - 0.5, z_max + 0.5, None, 0, 0],
        mode='lines',
        name='regionGrid: origin',
        legendgroup='regionGrid',
        visible=visible,
        showlegend=False,
        line=dict(color='rgba(40,40,40,0.85)', width=3),
        hoverinfo='skip',
    ))

def add_layer_inhabitance(fig: go.Figure, grids: dict[str, pd.DataFrame], visible: bool | str = True) -> None:
    grid = grids['inhabitance']
    fig.add_trace(go.Heatmap(
        z=grid.values,
        x=grid.columns,
        y=grid.index,
        name='inhabitance',
        legendgroup='inhabitance',
        visible=visible,
        showlegend=True,
        colorscale='YlOrRd',
        zmin=0,
        zmax=1,
        opacity=0.75,
        colorbar=dict(title='inhabitance', x=-0.12, y=0.82, len=0.32),
        hovertemplate='Chunk (%{x}, %{y})<br>inhabitance=%{z:.3f}<extra></extra>',
    ))

def add_layer_forgettable(fig: go.Figure, grids: dict[str, pd.DataFrame], visible: bool | str = 'legendonly') -> None:
    grid = grids['forgettable']
    z = _binary_mask(grid)
    if np.isnan(z).all():
        _add_legend_placeholder(fig, 'chunkForgettable', 'chunkForgettable', '#3a3a3a')
        return

    fig.add_trace(go.Heatmap(
        z=z,
        x=grid.columns,
        y=grid.index,
        name='chunkForgettable',
        legendgroup='chunkForgettable',
        visible=visible,
        showlegend=True,
        colorscale=[[0, '#3a3a3a'], [1, '#3a3a3a']],
        zmin=0, zmax=1,
        opacity=0.60,
        showscale=False,
        hovertemplate='Chunk (%{x}, %{y})<br>chunkForgettable=1<extra></extra>',
    ))

def add_layer_memorable(fig: go.Figure, grids: dict[str, pd.DataFrame], visible: bool | str = 'legendonly') -> None:
    grid = grids['memorable']
    z = _binary_mask(grid)
    if np.isnan(z).all():
        _add_legend_placeholder(fig, 'chunkMemorable', 'chunkMemorable', '#bfbfbf')
        return

    fig.add_trace(go.Heatmap(
        z=z,
        x=grid.columns,
        y=grid.index,
        name='chunkMemorable',
        legendgroup='chunkMemorable',
        visible=visible,
        showlegend=True,
        colorscale=[[0, '#bfbfbf'], [1, '#bfbfbf']],
        zmin=0, zmax=1,
        opacity=0.60,
        showscale=False,
        hovertemplate='Chunk (%{x}, %{y})<br>chunkMemorable=1<extra></extra>',
    ))

def add_layer_renewal_rank(fig: go.Figure, grids: dict[str, pd.DataFrame], visible: bool | str = 'legendonly') -> None:
    grid = grids['renewalRank']
    arr = grid.values.astype(float)
    if np.isnan(arr).all():
        _add_legend_placeholder(fig, 'renewalRank', 'renewalRank', '#4c78a8')
        return

    zmin = float(np.nanmin(arr)) if np.isfinite(arr).any() else 0.0
    zmax = float(np.nanmax(arr)) if np.isfinite(arr).any() else 1.0

    fig.add_trace(go.Heatmap(
        z=arr,
        x=grid.columns,
        y=grid.index,
        name='renewalRank',
        legendgroup='renewalRank',
        visible=visible,
        showlegend=True,
        colorscale='Viridis',
        zmin=zmin,
        zmax=zmax,
        opacity=0.75,
        colorbar=dict(title='renewalRank', x=-0.12, y=0.43, len=0.32),
        hovertemplate='Chunk (%{x}, %{y})<br>renewalRank=%{z}<extra></extra>',
    ))

def add_layer_renewal_action(fig: go.Figure, world_df: pd.DataFrame, all_actions: list[str], visible: bool | str = 'legendonly') -> None:
    """One marker trace per renewalAction category, with placeholders for missing categories."""
    d = world_df[world_df['renewalAction'] != 'NONE'].copy()
    present = set(d['renewalAction'].unique().tolist()) if not d.empty else set()
    palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#17becf']

    for i, action in enumerate(all_actions):
        color = palette[i % len(palette)]
        if action in present:
            subset = d[d['renewalAction'] == action]
            fig.add_trace(go.Scattergl(
                x=subset['x'],
                y=subset['z'],
                mode='markers',
                name=f'renewalAction: {action}',
                legendgroup='renewalAction',
                visible=visible,
                showlegend=True,
                marker=dict(symbol='square', size=7, color=color, opacity=0.9),
                hovertemplate='Chunk (%{x}, %{y})<br>renewalAction=' + action + '<extra></extra>',
            ))
        else:
            _add_legend_placeholder(fig, f'renewalAction: {action}', 'renewalAction', color)

def add_layer_dominant_stone_effect(fig: go.Figure, world_df: pd.DataFrame, all_effects: list[str], visible: bool | str = 'legendonly') -> None:
    d = world_df[world_df['dominantStoneEffect'] != 'NONE'].copy()
    if d.empty:
        _add_legend_placeholder(fig, 'dominantStoneEffect', 'dominantStoneEffect', '#9467bd')
        return

    categories = sorted(d['dominantStoneEffect'].unique().tolist())
    cat_to_idx = {cat: i for i, cat in enumerate(categories)}
    d['effect_idx'] = d['dominantStoneEffect'].map(cat_to_idx).astype(float)

    if len(categories) == 1:
        colorscale = [[0.0, '#9467bd'], [1.0, '#9467bd']]
        tickvals = [0]
    else:
        palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#17becf']
        colorscale = []
        for i, _cat in enumerate(categories):
            p = i / (len(categories) - 1)
            c = palette[i % len(palette)]
            colorscale.append([p, c])
        tickvals = list(range(len(categories)))

    fig.add_trace(go.Scattergl(
        x=d['x'],
        y=d['z'],
        mode='markers',
        name='dominantStoneEffect',
        legendgroup='dominantStoneEffect',
        visible=visible,
        showlegend=True,
        marker=dict(
            symbol='square',
            size=7,
            color=d['effect_idx'],
            colorscale=colorscale,
            cmin=0,
            cmax=max(len(categories) - 1, 1),
            colorbar=dict(title='dominantStoneEffect', tickvals=tickvals, ticktext=categories, x=-0.12, y=0.12, len=0.28),
        ),
        customdata=d[['dominantStoneEffect']].to_numpy(),
        hovertemplate='Chunk (%{x}, %{y})<br>dominantStoneEffect=%{customdata[0]}<extra></extra>',
    ))


## Chapter 6 — Compose One Figure per World
Apply the same layer pipeline to each discovered world dataset.

In [ ]:
# ----------------------------------------------------------
# Layer orchestration and figure generation
# ----------------------------------------------------------

def build_world_figure(world_name: str, world_df: pd.DataFrame) -> go.Figure:
    grids = build_world_grids(world_df)
    fig = build_canvas(world_name, world_df)

    # Layer build-up order
    add_layer_chunk_exists(fig, world_df, visible='legendonly')
    add_layer_inhabitance(fig, grids, visible=True)
    add_layer_forgettable(fig, grids, visible='legendonly')
    add_layer_memorable(fig, grids, visible='legendonly')
    add_layer_renewal_rank(fig, grids, visible=True)
    add_layer_renewal_action(fig, world_df, all_actions=GLOBAL_RENEWAL_ACTIONS, visible='legendonly')
    add_layer_dominant_stone_effect(fig, world_df, all_effects=GLOBAL_STONE_EFFECTS, visible=True)
    add_layer_region_grid(fig, world_df, visible=True)

    return fig

WORLD_FIGURES = {}
for world_name, world_df in WORLD_DATASETS.items():
    WORLD_FIGURES[world_name] = build_world_figure(world_name, world_df)

print(f'Generated figures: {len(WORLD_FIGURES)}')
for world_name in WORLD_FIGURES:
    print('-', world_name)

In [ ]:
# Display all world figures (dynamic world count)
for world_name, fig in WORLD_FIGURES.items():
    print(f'\n=== {world_name} ===')
    fig.show()

## Chapter 7 — Validation Summary
World-level summary statistics from the prepared datasets.

In [ ]:
summary_rows = []
for world_name, world_df in WORLD_DATASETS.items():
    summary_rows.append({
        'world': world_name,
        'rows': len(world_df),
        'x_min': int(world_df['x'].min()),
        'x_max': int(world_df['x'].max()),
        'z_min': int(world_df['z'].min()),
        'z_max': int(world_df['z'].max()),
        'chunkMemorable': int((world_df['memorableFlag'] == 1).sum()),
        'chunkForgettable': int((world_df['forgettableFlag'] == 1).sum()),
        'renewalAction_non_none': int((world_df['renewalAction'] != 'NONE').sum()),
        'dominantStoneEffect_non_none': int((world_df['dominantStoneEffect'] != 'NONE').sum()),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('world').reset_index(drop=True)
summary_df